<a href="https://colab.research.google.com/github/Bash33-oss/gpu-programming-cuda-optimization/blob/main/GPU_Programming_and_CUDA_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Sun Nov 22 03:36:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P0             28W /   70W |     125MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install numba numpy

In [ ]:
!pip install numba numpy

import numpy as np
import time
from numba import cuda, float32, uint8

print("GPU:", cuda.get_current_device().name)
print("GPU is active:", cuda.is_available())

# ==========================================================
# PROJECT 1: GPU IMAGE PROCESSING
# RGB TO GRAYSCALE CONVERSION
# ==========================================================

@cuda.jit
def grayscale_kernel(image, output):
    y, x = cuda.grid(2)

    if y < image.shape[0] and x < image.shape[1]:

        r = image[y, x, 0]
        g = image[y, x, 1]
        b = image[y, x, 2]

        gray = 0.299 * r + 0.587 * g + 0.114 * b

        output[y, x] = gray


# ==========================================================
# PROJECT 2: GPU IMAGE PROCESSING
# 3x3 BOX BLUR
# ==========================================================

@cuda.jit
def blur_kernel(image, output):

    y, x = cuda.grid(2)

    height = image.shape[0]
    width = image.shape[1]

    if y < height and x < width:

        total = 0.0
        count = 0

        for dy in range(-1, 2):

            for dx in range(-1, 2):

                yy = y + dy
                xx = x + dx

                if yy >= 0 and yy < height:
                    if xx >= 0 and xx < width:

                        total += image[yy, xx]
                        count += 1

        output[y, x] = total / count


# ==========================================================
# PROJECT 3: GPU MATRIX MULTIPLICATION (GEMM)
# ==========================================================

@cuda.jit
def gemm_kernel(A, B, C):

    row, col = cuda.grid(2)

    if row < C.shape[0] and col < C.shape[1]:

        accumulator = 0.0

        for k in range(A.shape[1]):

            accumulator += A[row, k] * B[k, col]

        C[row, col] = accumulator


# ==========================================================
# PROJECT 4: PARALLEL SUM REDUCTION
# ==========================================================

@cuda.jit
def reduction_kernel(data, partial_sums):

    shared = cuda.shared.array(256, dtype=float32)

    tid = cuda.threadIdx.x

    idx = cuda.blockIdx.x * cuda.blockDim.x + tid

    value = 0.0

    if idx < data.size:

        value = data[idx]

    shared[tid] = value

    cuda.syncthreads()

    stride = 128

    while stride > 0:

        if tid < stride:

            shared[tid] += shared[tid + stride]

        cuda.syncthreads()

        stride //= 2

    if tid == 0:

        partial_sums[cuda.blockIdx.x] = shared[0]


# ==========================================================
# CPU REFERENCE IMPLEMENTATIONS
# ==========================================================

def cpu_grayscale(image):

    image = image.astype(np.float32)

    return (
        0.299 * image[:, :, 0]
        + 0.587 * image[:, :, 1]
        + 0.114 * image[:, :, 2]
    ).astype(np.uint8)


# ==========================================================
# GPU BENCHMARKING FUNCTION
# ==========================================================

def benchmark(name, function, repetitions=10):

    function()

    cuda.synchronize()

    start = time.perf_counter()

    for _ in range(repetitions):

        function()

    cuda.synchronize()

    elapsed = (time.perf_counter() - start) * 1000

    average = elapsed / repetitions

    print(f"{name}: {average:.4f} ms")


# ==========================================================
# EXECUTION
# ==========================================================

print("\n" + "=" * 50)
print("GPU PROGRAMMING PORTFOLIO")
print("=" * 50)


# ----------------------------------------------------------
# IMAGE PROCESSING
# ----------------------------------------------------------

height = 1024
width = 1024

rng = np.random.default_rng(42)

image = rng.integers(
    0,
    256,
    size=(height, width, 3),
    dtype=np.uint8
)

gpu_image = cuda.to_device(image)

gpu_gray = cuda.device_array(
    (height, width),
    dtype=np.uint8
)

gpu_blur = cuda.device_array(
    (height, width),
    dtype=np.uint8
)

threads = (16, 16)

blocks = (
    (height + 15) // 16,
    (width + 15) // 16
)

print("\n1. IMAGE PROCESSING")

grayscale_kernel[blocks, threads](
    gpu_image,
    gpu_gray
)

blur_kernel[blocks, threads](
    gpu_gray,
    gpu_blur
)

cuda.synchronize()

gpu_gray_result = gpu_gray.copy_to_host()

cpu_gray_result = cpu_grayscale(image)

error = np.max(
    np.abs(
        gpu_gray_result.astype(np.int16)
        - cpu_gray_result.astype(np.int16)
    )
)

print("Grayscale GPU execution successful")
print("Maximum pixel error:", error)

benchmark(
    "Grayscale kernel",
    lambda: grayscale_kernel[blocks, threads](
        gpu_image,
        gpu_gray
    )
)

benchmark(
    "3x3 Box Blur kernel",
    lambda: blur_kernel[blocks, threads](
        gpu_gray,
        gpu_blur
    )
)


# ----------------------------------------------------------
# MATRIX MULTIPLICATION
# ----------------------------------------------------------

print("\n2. MATRIX MULTIPLICATION (GEMM)")

n = 512

A = rng.random(
    (n, n),
    dtype=np.float32
)

B = rng.random(
    (n, n),
    dtype=np.float32
)

gpu_A = cuda.to_device(A)

gpu_B = cuda.to_device(B)

gpu_C = cuda.device_array(
    (n, n),
    dtype=np.float32
)

gemm_kernel[blocks, threads](
    gpu_A,
    gpu_B,
    gpu_C
)

cuda.synchronize()

gpu_result = gpu_C.copy_to_host()

cpu_result = np.matmul(A, B)

gemm_error = np.max(
    np.abs(gpu_result - cpu_result)
)

print("Matrix multiplication completed")
print("Matrix dimensions:", n, "x", n)
print("Maximum numerical error:", gemm_error)

benchmark(
    "Naive GPU GEMM",
    lambda: gemm_kernel[blocks, threads](
        gpu_A,
        gpu_B,
        gpu_C
    )
)


# ----------------------------------------------------------
# PARALLEL REDUCTION
# ----------------------------------------------------------

print("\n3. PARALLEL SUM REDUCTION")

data = rng.random(
    1_000_000,
    dtype=np.float32
)

gpu_data = cuda.to_device(data)

threads_reduction = 256

blocks_reduction = (
    data.size + threads_reduction - 1
) // threads_reduction

gpu_partial_sums = cuda.device_array(
    blocks_reduction,
    dtype=np.float32
)

reduction_kernel[
    blocks_reduction,
    threads_reduction
](
    gpu_data,
    gpu_partial_sums
)

cuda.synchronize()

partial_results = gpu_partial_sums.copy_to_host()

gpu_sum = np.sum(partial_results)

cpu_sum = np.sum(data)

reduction_error = abs(gpu_sum - cpu_sum)

print("Reduction completed")
print("Elements processed:", data.size)
print("GPU sum:", gpu_sum)
print("CPU sum:", cpu_sum)
print("Absolute error:", reduction_error)

benchmark(
    "GPU Parallel Reduction",
    lambda: reduction_kernel[
        blocks_reduction,
        threads_reduction
    ](
        gpu_data,
        gpu_partial_sums
    )
)


# ==========================================================
# PROJECT SUMMARY
# ==========================================================

print("\n" + "=" * 50)
print("PROJECT EXECUTION SUMMARY")
print("=" * 50)

print("GPU:", cuda.get_current_device().name)

print("Image Processing: SUCCESS")

print("Grayscale Conversion: SUCCESS")

print("3x3 Box Blur: SUCCESS")

print("Matrix Multiplication: SUCCESS")

print("Parallel Reduction: SUCCESS")

print("\nAll GPU kernels executed successfully!")


GPU: Tesla T4
GPU is active: True

GPU PROGRAMMING PORTFOLIO

1. IMAGE PROCESSING
Grayscale GPU execution successful
Maximum pixel error: 1
Grayscale kernel: 0.2251 ms
3x3 Box Blur kernel: 0.7169 ms

2. MATRIX MULTIPLICATION (GEMM)
Matrix multiplication completed
Matrix dimensions: 512 x 512
Maximum numerical error: 9.1552734e-05
Naive GPU GEMM: 6.6977 ms

3. PARALLEL SUM REDUCTION
Reduction completed
Elements processed: 1000000
GPU sum: 500141.1
CPU sum: 500141.1
Absolute error: 0.0
GPU Parallel Reduction: 0.0964 ms

PROJECT EXECUTION SUMMARY
GPU: Tesla T4
Image Processing: SUCCESS
Grayscale Conversion: SUCCESS
3x3 Box Blur: SUCCESS
Matrix Multiplication: SUCCESS
Parallel Reduction: SUCCESS

All GPU kernels executed successfully!


In [ ]:
import numpy as np
from numba import cuda, float32
import time

# ==========================================
# OPTIMIZED GPU MATRIX MULTIPLICATION
# USING SHARED MEMORY
# ==========================================

@cuda.jit
def gemm_shared(A, B, C):

    TILE_SIZE = 16

    # Shared memory tiles
    shared_A = cuda.shared.array(
        shape=(16, 16),
        dtype=float32
    )

    shared_B = cuda.shared.array(
        shape=(16, 16),
        dtype=float32
    )

    row = cuda.blockIdx.y * TILE_SIZE + cuda.threadIdx.y
    col = cuda.blockIdx.x * TILE_SIZE + cuda.threadIdx.x

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    accumulator = 0.0

    # Iterate through matrix tiles
    for tile in range(
        (A.shape[1] + TILE_SIZE - 1) // TILE_SIZE
    ):

        # Load A tile into shared memory
        if row < A.shape[0]:

            if tile * TILE_SIZE + tx < A.shape[1]:

                shared_A[ty, tx] = A[
                    row,
                    tile * TILE_SIZE + tx
                ]

            else:

                shared_A[ty, tx] = 0.0

        else:

            shared_A[ty, tx] = 0.0

        # Load B tile into shared memory
        if col < B.shape[1]:

            if tile * TILE_SIZE + ty < B.shape[0]:

                shared_B[ty, tx] = B[
                    tile * TILE_SIZE + ty,
                    col
                ]

            else:

                shared_B[ty, tx] = 0.0

        else:

            shared_B[ty, tx] = 0.0

        # Synchronize threads
        cuda.syncthreads()

        # Compute tile product
        for k in range(TILE_SIZE):

            accumulator += (
                shared_A[ty, k] *
                shared_B[k, tx]
            )

        # Synchronize before loading next tile
        cuda.syncthreads()

    # Store final result
    if row < C.shape[0] and col < C.shape[1]:

        C[row, col] = accumulator


print("Shared-memory GEMM kernel defined successfully!")



Shared-memory GEMM kernel defined successfully!


In [ ]:
# ==========================================
# TEST OPTIMIZED GEMM
# ==========================================

n = 512

rng = np.random.default_rng(42)

A = rng.random((n, n), dtype=np.float32)

B = rng.random((n, n), dtype=np.float32)

# CPU reference
cpu_start = time.perf_counter()

C_cpu = np.matmul(A, B)

cpu_time = time.perf_counter() - cpu_start

# Transfer matrices to GPU
A_gpu = cuda.to_device(A)

B_gpu = cuda.to_device(B)

C_gpu = cuda.device_array(
    (n, n),
    dtype=np.float32
)

# Configure GPU
threads = (16, 16)

blocks = (
    (n + 15) // 16,
    (n + 15) // 16
)

# Run optimized kernel
gemm_shared[blocks, threads](
    A_gpu,
    B_gpu,
    C_gpu
)

cuda.synchronize()

# Retrieve result
C_result = C_gpu.copy_to_host()

# Validate
error = np.max(
    np.abs(C_cpu - C_result)
)

print("Matrix size:", n, "x", n)

print("Maximum numerical error:", error)

print("CPU execution time:", cpu_time * 1000, "ms")

print("Optimized GPU GEMM completed successfully!")

Matrix size: 512 x 512
Maximum numerical error: 9.1552734e-05
CPU execution time: 2.711396999984572 ms
Optimized GPU GEMM completed successfully!


In [ ]:
# ==========================================
# BENCHMARK NAIVE VS OPTIMIZED GEMM
# ==========================================

def benchmark_gpu(kernel, A_gpu, B_gpu, C_gpu, blocks, threads, repetitions=10):

    # Warm-up
    kernel[blocks, threads](A_gpu, B_gpu, C_gpu)

    cuda.synchronize()

    start = time.perf_counter()

    for _ in range(repetitions):

        kernel[blocks, threads](
            A_gpu,
            B_gpu,
            C_gpu
        )

    cuda.synchronize()

    elapsed = time.perf_counter() - start

    return (elapsed / repetitions) * 1000


# Benchmark naive kernel
naive_time = benchmark_gpu(
    gemm_kernel,
    A_gpu,
    B_gpu,
    C_gpu,
    blocks,
    threads
)

# Benchmark shared memory kernel
optimized_time = benchmark_gpu(
    gemm_shared,
    A_gpu,
    B_gpu,
    C_gpu,
    blocks,
    threads
)

print("=" * 40)

print("GEMM PERFORMANCE COMPARISON")

print("=" * 40)

print("Naive GEMM:", naive_time, "ms")

print("Shared-memory GEMM:", optimized_time, "ms")

if optimized_time > 0:

    speedup = naive_time / optimized_time

    print("Speedup:", speedup, "x")



GEMM PERFORMANCE COMPARISON
Naive GEMM: 6.533200800004124 ms
Shared-memory GEMM: 5.522653200023342 ms
Speedup: 1.1829822665628382 x
